In [6]:
# ══════════════════════════════════════════════════════════════
# SETUP CELL — GitHub → DQN (flat zip, files directly inside)
# ══════════════════════════════════════════════════════════════
import subprocess, shutil, os, zipfile, sys

# Step 1 — Clone repo
if os.path.exists('/content/repo'):
    shutil.rmtree('/content/repo')

print('Cloning from GitHub...')
subprocess.run([
    'git', 'clone',
    'https://github.com/maham-bhatti/Traffic-Control-System.git',
    '/content/repo'
], check=True)
print('✓ Repo cloned')

# Step 2 — Extract flat zip directly to /content
zip_path = '/content/repo/DQN.zip'
print('\nExtracting DQN.zip...')
with zipfile.ZipFile(zip_path, 'r') as z:
    print('── Files inside zip ──')
    for name in sorted(z.namelist()):
        print(f'  {name}')
    z.extractall('/content/')   # ← flat zip so extract straight to /content
print('✓ Extracted directly to /content')

# Step 3 — Python path
sys.path.insert(0, '/content')

# Step 4 — Verify
REQUIRED = [
    'gcn_encoder.py', 'ma_dqn.py', 'traci_env_dqn.py',
    'manhattan.net.xml', 'run.sumocfg',
    'final_routes.rou.xml', 'low_routes.rou.xml',
    'medium_routes.rou.xml', 'high_routes.rou.xml',
]
print('\n── Required files check ──')
all_ok = True
for f in REQUIRED:
    exists = os.path.exists(f'/content/{f}')
    size   = f'{os.path.getsize(f"/content/{f}")/1024:.1f} KB' if exists else ''
    if not exists: all_ok = False
    print(f'  {"✓" if exists else "✗ MISSING":<12} {f:<35} {size}')

print()
print('✅ All set! Now run Cell 1.' if all_ok else '❌ Some files missing — check zip contents above!')

Cloning from GitHub...
✓ Repo cloned

Extracting DQN.zip...
── Files inside zip ──
  final_routes.rou.xml
  gcn_encoder.py
  generate_density_routes.py
  high_routes.rou.xml
  low_routes.rou.xml
  ma_dqn.py
  manhattan.net.xml
  manhattan_labels.poi.xml
  medium_routes.rou.xml
  run.sumocfg
  test_env.py
  traci_env_dqn.py
✓ Extracted directly to /content

── Required files check ──
  ✓            gcn_encoder.py                      9.5 KB
  ✓            ma_dqn.py                           16.0 KB
  ✓            traci_env_dqn.py                    22.7 KB
  ✓            manhattan.net.xml                   56.4 KB
  ✓            run.sumocfg                         0.3 KB
  ✓            final_routes.rou.xml                13.8 KB
  ✓            low_routes.rou.xml                  14.0 KB
  ✓            medium_routes.rou.xml               14.0 KB
  ✓            high_routes.rou.xml                 14.0 KB

✅ All set! Now run Cell 1.


# GCN + Multi-Agent DQN
## Manhattan 4x4 Traffic Signal Control — Kaggle Training

**Algorithm**: Deep Q-network (64→128→action), replay buffer per agent, target network updated every 200 steps.

**Training structure — identical to MAPPO and Q-Learning for fair paper comparison:**

| Stage | Density | Episodes | Traffic Volume | Purpose |
|-------|---------|----------|----------------|---------|
| 0 | Baseline (default routes) | 5 test / 300 full | Mixed | Agent learns fundamentals |
| 1 | Low (off-peak) | 5 test / 200 full | ~824 veh/hr | Quiet roads |
| 2 | Medium (mid-day) | 5 test / 200 full | ~1685 veh/hr | Moderate traffic |
| 3 | High (peak-hour) | 5 test / 400 full | ~3376 veh/hr | Hardest, most important |

**Files required in Kaggle dataset:**
`gcn_encoder.py`, `ma_dqn.py`, `traci_env_dqn.py`, `manhattan.net.xml`,
`run.sumocfg`, `final_routes.rou.xml`, `low_routes.rou.xml`, `medium_routes.rou.xml`, `high_routes.rou.xml`

| Cell | Purpose |
|------|---------|
| 1 | Install SUMO |
| 2 | Copy dataset files |
| 3 | Verify all required files |
| 4 | Install Python packages |
| 5 | Environment test (50 steps) |
| 6 | **Stage 0** — Baseline test (5 ep) then full (300 ep) |
| 7 | **Stage 1** — Low density test (5 ep) then full (200 ep) |
| 8 | **Stage 2** — Medium density test (5 ep) then full (200 ep) |
| 9 | **Stage 3** — High density test (5 ep) then full (400 ep) |
| 10 | Evaluate on all 3 densities |
| 11 | Plot training curves across all stages |

In [7]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Install SUMO (Internet must be ON, takes ~2-3 min)
# ══════════════════════════════════════════════════════════════
import subprocess, sys, os

print('Adding SUMO stable PPA...')
subprocess.run(['add-apt-repository', 'ppa:sumo/stable', '-y'],
               capture_output=True, text=True)

print('Updating apt...')
subprocess.run(['apt-get', 'update', '-q'],
               capture_output=True, text=True)

print('Installing SUMO...')
r = subprocess.run(
    ['apt-get', 'install', '-y', 'sumo', 'sumo-tools', 'sumo-doc'],
    capture_output=True, text=True
)
print(r.stdout[-500:] if r.stdout else '')
if r.returncode != 0:
    print('apt stderr:', r.stderr[-300:])

# Set SUMO_HOME BEFORE importing traci anywhere
os.environ['SUMO_HOME'] = '/usr/share/sumo'
if '/usr/share/sumo/tools' not in sys.path:
    sys.path.insert(0, '/usr/share/sumo/tools')

# Kill any stale SUMO processes
subprocess.run(['pkill', '-9', '-f', 'sumo'],  capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'traci'], capture_output=True)

# Force clear any cached traci modules
for mod in list(sys.modules.keys()):
    if 'traci' in mod or 'sumo' in mod:
        del sys.modules[mod]

# Verify installation
ver = subprocess.run(['sumo', '--version'], capture_output=True, text=True)
if 'SUMO' in ver.stdout:
    print('\n✓ SUMO installed:', ver.stdout.split('\n')[0])
else:
    print('✗ ERROR: SUMO not found. Check stderr below:')
    print(ver.stderr)


Adding SUMO stable PPA...
Updating apt...
Installing SUMO...
k

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link



✓ SUMO installed: Eclipse SUMO sumo 1.26.0


In [9]:
# CELL 3: Verify all required files exist
import os
WORK_DIR = '/content'
REQUIRED = {
    'gcn_encoder.py'        : 'GCN encoder — shared across all 3 algorithms',
    'ma_dqn.py'             : 'DQN controller (DeepQNetwork + ReplayBuffer + MADQNController)',
    'traci_env_dqn.py'      : 'SUMO environment + DQN-specific training loop',
    'manhattan.net.xml'     : 'Road network — 12 TLS junctions',
    'run.sumocfg'           : 'SUMO base config',
    'final_routes.rou.xml'  : 'Baseline route file (Stage 0)',
    'low_routes.rou.xml'    : 'Off-peak routes (~824 veh/hr) — Stage 1',
    'medium_routes.rou.xml' : 'Moderate routes (~1685 veh/hr) — Stage 2',
    'high_routes.rou.xml'   : 'Peak-hour routes (~3376 veh/hr) — Stage 3',
}
all_ok = True
print(f'{"FILE":<30} STATUS')
print('-' * 65)
for fname, desc in REQUIRED.items():
    path = os.path.join(WORK_DIR, fname)
    ok   = os.path.exists(path)
    size = f'{os.path.getsize(path):,} B' if ok else ''
    if not ok: all_ok = False
    print(f'{fname:<30} {"OK  " + size if ok else "MISSING"}  ({desc})')
print()
if all_ok:
    print('All files present — safe to proceed')
else:
    print('ERROR: Upload missing files to your Kaggle dataset then re-run Cell 2')

FILE                           STATUS
-----------------------------------------------------------------
gcn_encoder.py                 OK  9,686 B  (GCN encoder — shared across all 3 algorithms)
ma_dqn.py                      OK  16,421 B  (DQN controller (DeepQNetwork + ReplayBuffer + MADQNController))
traci_env_dqn.py               OK  23,209 B  (SUMO environment + DQN-specific training loop)
manhattan.net.xml              OK  57,729 B  (Road network — 12 TLS junctions)
run.sumocfg                    OK  277 B  (SUMO base config)
final_routes.rou.xml           OK  14,104 B  (Baseline route file (Stage 0))
low_routes.rou.xml             OK  14,300 B  (Off-peak routes (~824 veh/hr) — Stage 1)
medium_routes.rou.xml          OK  14,317 B  (Moderate routes (~1685 veh/hr) — Stage 2)
high_routes.rou.xml            OK  14,354 B  (Peak-hour routes (~3376 veh/hr) — Stage 3)

All files present — safe to proceed


In [10]:
# CELL 4: Install Python packages
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'numpy', 'pandas', 'matplotlib'])
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {"CUDA" if torch.cuda.is_available() else "CPU"} ({gpu})')
print('Packages ready')

PyTorch : 2.10.0+cpu
Device  : CPU (no GPU)
Packages ready


In [11]:
# CELL 5: Environment test — 50 random steps on each density
# Confirms SUMO + reward + all 3 route files are working before training
import os, sys, numpy as np
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from gcn_encoder import REAL_JUNCTIONS, RAW_OBS_DIM, ACT_DIM
from traci_env_dqn import SUMOEnv

all_passed = True
for density, label in [(None, 'Baseline'), ('low', 'Low'), ('medium', 'Medium'), ('high', 'High')]:
    print(f'Testing {label} density...')
    try:
        env = SUMOEnv(warmup=50, ep_duration=100, density=density)
        obs = env.reset()
        shape_ok = all(obs[jid].shape[0] == RAW_OBS_DIM[jid] for jid in REAL_JUNCTIONS)
        rewards = []
        for _ in range(50):
            acts = {jid: np.random.randint(0, ACT_DIM[jid]) for jid in REAL_JUNCTIONS}
            obs, rews, done, _ = env.step(acts)
            rewards.extend(rews.values())
            if done: break
        env.close()
        r_min, r_max = min(rewards), max(rewards)
        in_range = r_min >= -1.001 and r_max <= 0.001
        status = 'PASSED' if (shape_ok and in_range) else 'FAILED'
        if not (shape_ok and in_range): all_passed = False
        print(f'  {status} | reward=[{r_min:.3f}, {r_max:.3f}] | shape_ok={shape_ok}')
    except Exception as e:
        print(f'  ERROR: {e}')
        all_passed = False

print()
print('All densities OK — proceed to Stage 0' if all_passed else 'Fix errors above before training')

Testing Baseline density...
 Retrying in 1 seconds
  PASSED | reward=[-0.185, -0.000] | shape_ok=True
Testing Low density...
  [SUMOEnv] density=low → low_routes.rou.xml
 Retrying in 1 seconds
  PASSED | reward=[-0.185, -0.000] | shape_ok=True
Testing Medium density...
  [SUMOEnv] density=medium → medium_routes.rou.xml
 Retrying in 1 seconds
  PASSED | reward=[-0.185, -0.000] | shape_ok=True
Testing High density...
  [SUMOEnv] density=high → high_routes.rou.xml
 Retrying in 1 seconds
  PASSED | reward=[-0.185, -0.000] | shape_ok=True

All densities OK — proceed to Stage 0


---
## Stage 0 — Baseline Training (default routes)
Agent learns basic traffic signal control on the default route file.
**Run Cell 6a first (5 episodes).** If it passes, run Cell 6b (300 episodes).

In [ ]:
# CELL 6: Stage 0 — Full baseline training (300 episodes)
# Only run after Cell 6a completes without errors
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_dqn import train
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Stage 0 — Baseline | 600 episodes | density=None (default routes)')
print()
ctrl = train(
    n_episodes  = 600,
    device      = DEVICE,
    save_dir    = '/content/ckpt_dqn_base',
    use_gui     = False,
    curriculum  = False,
    density     = None,
)
print('Stage 0 complete — proceed to Stage 1')

Stage 0 — Baseline | 300 episodes | density=None (default routes)

  GCN-MA-DQN Training  |  device=cpu  |  episodes=600
  Algorithm  : DQN — deep Q-network + replay buffer + target network
  Reward     : -(0.5·NormQueue + 0.3·NormDelay + 0.2·NormWait)
  Update     : Batch TD, replay buffer per agent, target network
  Curriculum : OFF

  MADQNController ready
  Algorithm   : DQN  |  Q-network: Deep MLP (64→128→act)
  Junctions   : 12
  GCN params  : 6,560   DQN per agent: 11,330   Total: 278,480
  Buffer size : 10,000 transitions per agent
  Target update: every 200 steps (hard copy)
  Epsilon     : 1.0 → 0.05  (decay 0.995/episode)
 Retrying in 1 seconds


KeyboardInterrupt: 

In [13]:
import os, zipfile

CKPT_DIR       = "/content/ckpt_dqn_base"
CKPT_FINAL_DIR = "/content/ckpt_dqn_base_final"
OUTPUT_ZIP     = "/content/dqn_base_density_results.zip"

added = set()  # track added files to avoid duplicates

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for src_dir in [CKPT_DIR, CKPT_FINAL_DIR]:
        if not os.path.isdir(src_dir):
            print(f"[WARN] Not found: {src_dir}")
            continue
        for root, _, files in os.walk(src_dir):
            for fname in files:
                full_path = os.path.join(root, fname)
                arcname   = os.path.relpath(full_path, "/content")
                if arcname not in added:          # ← fixes duplicate warning
                    zf.write(full_path, arcname)
                    added.add(arcname)
                    print(f"  + {arcname}")
                else:
                    print(f"  [SKIP duplicate] {arcname}")

size_kb = os.path.getsize(OUTPUT_ZIP) / 1024
print(f"\n✅ Zip ready: {OUTPUT_ZIP}  ({size_kb:.1f} KB)")
print("👉 Go to the OUTPUT panel on the right → click the download icon next to the zip file.")

[WARN] Not found: /content/ckpt_dqn_base
[WARN] Not found: /content/ckpt_dqn_base_final

✅ Zip ready: /content/dqn_base_density_results.zip  (0.0 KB)
👉 Go to the OUTPUT panel on the right → click the download icon next to the zip file.


---
## Stage 1 — Low Density (off-peak, ~824 veh/hr)
Resumes from Stage 0 checkpoint. Agent trains on quiet roads.
**Run Cell 7a first (5 episodes).** If it passes, run Cell 7b (200 episodes).

In [14]:
# CELL 7: Stage 1 — Full low density training (200 episodes)
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_dqn import train
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
resume = '/content/ckpt_dqn_base_best'
if not os.path.exists(resume):
    resume = '/content/ckpt_dqn_base'
print(f'Stage 1 — Low density | 200 episodes | resume from: {resume}')
print()
ctrl = train(
    n_episodes  = 800,
    device      = DEVICE,
    save_dir    = '/content/ckpt_dqn_low',
    resume_from = resume,
    use_gui     = False,
    curriculum  = False,
    density     = 'low',
)
print('Stage 1 complete — proceed to Stage 2')

Stage 1 — Low density | 200 episodes | resume from: /content/ckpt_dqn_base

  GCN-MA-DQN Training  |  device=cpu  |  episodes=800
  Algorithm  : DQN — deep Q-network + replay buffer + target network
  Reward     : -(0.5·NormQueue + 0.3·NormDelay + 0.2·NormWait)
  Update     : Batch TD, replay buffer per agent, target network
  Curriculum : OFF

  MADQNController ready
  Algorithm   : DQN  |  Q-network: Deep MLP (64→128→act)
  Junctions   : 12
  GCN params  : 6,560   DQN per agent: 11,330   Total: 278,480
  Buffer size : 10,000 transitions per agent
  Target update: every 200 steps (hard copy)
  Epsilon     : 1.0 → 0.05  (decay 0.995/episode)
  Loaded <- /content/ckpt_dqn_base/
  [SUMOEnv] density=low → low_routes.rou.xml
 Retrying in 1 seconds


KeyboardInterrupt: 

In [15]:
import os, zipfile

CKPT_DIR       = "/content/ckpt_dqn_low"
CKPT_FINAL_DIR = "/content/ckpt_dqn_low_final"
OUTPUT_ZIP     = "/content/dqn_low_density_results.zip"

added = set()  # track added files to avoid duplicates

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for src_dir in [CKPT_DIR, CKPT_FINAL_DIR]:
        if not os.path.isdir(src_dir):
            print(f"[WARN] Not found: {src_dir}")
            continue
        for root, _, files in os.walk(src_dir):
            for fname in files:
                full_path = os.path.join(root, fname)
                arcname   = os.path.relpath(full_path, "/content")
                if arcname not in added:          # ← fixes duplicate warning
                    zf.write(full_path, arcname)
                    added.add(arcname)
                    print(f"  + {arcname}")
                else:
                    print(f"  [SKIP duplicate] {arcname}")

size_kb = os.path.getsize(OUTPUT_ZIP) / 1024
print(f"\n✅ Zip ready: {OUTPUT_ZIP}  ({size_kb:.1f} KB)")
print("👉 Go to the OUTPUT panel on the right → click the download icon next to the zip file.")

[WARN] Not found: /content/ckpt_dqn_low
[WARN] Not found: /content/ckpt_dqn_low_final

✅ Zip ready: /content/dqn_low_density_results.zip  (0.0 KB)
👉 Go to the OUTPUT panel on the right → click the download icon next to the zip file.


---
## Stage 2 — Medium Density (mid-day, ~1685 veh/hr)
Resumes from Stage 1 checkpoint. Agent trains on moderate traffic.
**Run Cell 8a first (5 episodes).** If it passes, run Cell 8b (200 episodes).

In [16]:
# CELL 8: Stage 2 — Full medium density training (200 episodes)
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_dqn import train
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
resume = '/content/ckpt_dqn_low_best'
if not os.path.exists(resume):
    resume = '/content/ckpt_dqn_low'
print(f'Stage 2 — Medium density | 200 episodes | resume from: {resume}')
print()
ctrl = train(
    n_episodes  = 900,
    device      = DEVICE,
    save_dir    = '/content/ckpt_dqn_med',
    resume_from = resume,
    use_gui     = False,
    curriculum  = False,
    density     = 'medium',
)
print('Stage 2 complete — proceed to Stage 3')

Stage 2 — Medium density | 200 episodes | resume from: /content/ckpt_dqn_low

  GCN-MA-DQN Training  |  device=cpu  |  episodes=900
  Algorithm  : DQN — deep Q-network + replay buffer + target network
  Reward     : -(0.5·NormQueue + 0.3·NormDelay + 0.2·NormWait)
  Update     : Batch TD, replay buffer per agent, target network
  Curriculum : OFF

  MADQNController ready
  Algorithm   : DQN  |  Q-network: Deep MLP (64→128→act)
  Junctions   : 12
  GCN params  : 6,560   DQN per agent: 11,330   Total: 278,480
  Buffer size : 10,000 transitions per agent
  Target update: every 200 steps (hard copy)
  Epsilon     : 1.0 → 0.05  (decay 0.995/episode)
  Loaded <- /content/ckpt_dqn_low/
  [SUMOEnv] density=medium → medium_routes.rou.xml


KeyError: 65

In [17]:
import os, zipfile

CKPT_DIR       = "/content/ckpt_dqn_med"
CKPT_FINAL_DIR = "/content/ckpt_dqn_med_final"
OUTPUT_ZIP     = "/content/dqn_med_density_results.zip"

added = set()  # track added files to avoid duplicates

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for src_dir in [CKPT_DIR, CKPT_FINAL_DIR]:
        if not os.path.isdir(src_dir):
            print(f"[WARN] Not found: {src_dir}")
            continue
        for root, _, files in os.walk(src_dir):
            for fname in files:
                full_path = os.path.join(root, fname)
                arcname   = os.path.relpath(full_path, "/content")
                if arcname not in added:          # ← fixes duplicate warning
                    zf.write(full_path, arcname)
                    added.add(arcname)
                    print(f"  + {arcname}")
                else:
                    print(f"  [SKIP duplicate] {arcname}")

size_kb = os.path.getsize(OUTPUT_ZIP) / 1024
print(f"\n✅ Zip ready: {OUTPUT_ZIP}  ({size_kb:.1f} KB)")
print("👉 Go to the OUTPUT panel on the right → click the download icon next to the zip file.")

[WARN] Not found: /content/ckpt_dqn_med
[WARN] Not found: /content/ckpt_dqn_med_final

✅ Zip ready: /content/dqn_med_density_results.zip  (0.0 KB)
👉 Go to the OUTPUT panel on the right → click the download icon next to the zip file.


---
## Stage 3 — High Density (peak-hour, ~3376 veh/hr)
Resumes from Stage 2 checkpoint. Hardest and most important stage.
This is the final checkpoint used for paper evaluation.
**Run Cell 9a first (5 episodes).** If it passes, run Cell 9b (400 episodes).

In [18]:
# CELL 9: Stage 3 — Full high density training (400 episodes)
# This is the most important stage — 400 episodes because peak-hour is hardest
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_dqn import train
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
resume = '/content/ckpt_dqn_med_best'
if not os.path.exists(resume):
    resume = '/content/ckpt_dqn_med'
print(f'Stage 3 — High density | 400 episodes | resume from: {resume}')
print()
ctrl = train(
    n_episodes  = 1200,
    device      = DEVICE,
    save_dir    = '/content/ckpt_dqn_high',
    resume_from = resume,
    use_gui     = False,
    curriculum  = False,
    density     = 'high',
)
print('Stage 3 complete — proceed to evaluation')

Stage 3 — High density | 400 episodes | resume from: /content/ckpt_dqn_med

  GCN-MA-DQN Training  |  device=cpu  |  episodes=1200
  Algorithm  : DQN — deep Q-network + replay buffer + target network
  Reward     : -(0.5·NormQueue + 0.3·NormDelay + 0.2·NormWait)
  Update     : Batch TD, replay buffer per agent, target network
  Curriculum : OFF

  MADQNController ready
  Algorithm   : DQN  |  Q-network: Deep MLP (64→128→act)
  Junctions   : 12
  GCN params  : 6,560   DQN per agent: 11,330   Total: 278,480
  Buffer size : 10,000 transitions per agent
  Target update: every 200 steps (hard copy)
  Epsilon     : 1.0 → 0.05  (decay 0.995/episode)
  Loaded <- /content/ckpt_dqn_med/
  [SUMOEnv] density=high → high_routes.rou.xml


FatalTraCIError: Received answer 163 for command 127.

In [19]:
import os, zipfile

CKPT_DIR       = "/content/ckpt_dqn_high"
CKPT_FINAL_DIR = "/content/ckpt_dqn_high_final"
OUTPUT_ZIP     = "/content/dqn_high_density_results.zip"

added = set()  # track added files to avoid duplicates

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for src_dir in [CKPT_DIR, CKPT_FINAL_DIR]:
        if not os.path.isdir(src_dir):
            print(f"[WARN] Not found: {src_dir}")
            continue
        for root, _, files in os.walk(src_dir):
            for fname in files:
                full_path = os.path.join(root, fname)
                arcname   = os.path.relpath(full_path, "/content")
                if arcname not in added:          # ← fixes duplicate warning
                    zf.write(full_path, arcname)
                    added.add(arcname)
                    print(f"  + {arcname}")
                else:
                    print(f"  [SKIP duplicate] {arcname}")

size_kb = os.path.getsize(OUTPUT_ZIP) / 1024
print(f"\n✅ Zip ready: {OUTPUT_ZIP}  ({size_kb:.1f} KB)")
print("👉 Go to the OUTPUT panel on the right → click the download icon next to the zip file.")

[WARN] Not found: /content/ckpt_dqn_high
[WARN] Not found: /content/ckpt_dqn_high_final

✅ Zip ready: /content/dqn_high_density_results.zip  (0.0 KB)
👉 Go to the OUTPUT panel on the right → click the download icon next to the zip file.


---
## Evaluation — Same model tested on all 3 densities
Loads the best high-density checkpoint and evaluates on low, medium, and high.
This is the table you report in your paper.

In [ ]:
# CELL 10: Evaluate trained DQN on all 3 densities
# Uses best checkpoint from Stage 3 (high density)
# epsilon forced to 0 during evaluation = pure greedy argmax Q
import os, sys, torch, numpy as np
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_dqn import _evaluate
from ma_dqn import MADQNController
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load best Stage 3 checkpoint
ckpt = '/content/ckpt_dqn_high_best'
if not os.path.exists(ckpt):
    ckpt = '/content/ckpt_dqn_high_final'
if not os.path.exists(ckpt):
    ckpt = '/content/ckpt_dqn_high'
print(f'Loading checkpoint: {ckpt}')
ctrl = MADQNController(device=DEVICE)
ctrl.load(ckpt)
print()

print('=' * 55)
print('  GCN-DQN Evaluation — same model, all 3 densities')
print('  (epsilon=0, pure greedy, 5 reps each)')
print('=' * 55)
for density, label in [
    ('low',    'Stage 1 — Low density    (~824  veh/hr)'),
    ('medium', 'Stage 2 — Medium density (~1685 veh/hr)'),
    ('high',   'Stage 3 — High density   (~3376 veh/hr)'),
]:
    print(f'\n{label}')
    _evaluate(ctrl, n_reps=5, use_gui=False, density=density)
print()
print('Copy this table into your paper for comparison with MAPPO and Q-Learning')

---
## Training Curves — All 4 Stages

In [ ]:
# CELL 11: Plot training curves for all 4 stages
import pandas as pd, matplotlib.pyplot as plt, os

STAGES = [
    ('ckpt_dqn_base', 'Stage 0: Baseline',       '#888780'),
    ('ckpt_dqn_low',  'Stage 1: Low density',     '#1D9E75'),
    ('ckpt_dqn_med',  'Stage 2: Medium density',  '#378ADD'),
    ('ckpt_dqn_high', 'Stage 3: High density',    '#E24B4A'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('GCN + Multi-Agent DQN — Training Curves (All Stages)',
             fontsize=14, fontweight='bold')

for ckpt_name, label, color in STAGES:
    log_path = f'/content/{ckpt_name}/training_log.csv'
    if not os.path.exists(log_path):
        print(f'Skipping {label} — log not found at {log_path}')
        continue
    df = pd.read_csv(log_path)
    w  = max(1, len(df) // 20)

    # Reward
    axes[0,0].plot(df['episode'],
                   df['avg_reward'].rolling(w, min_periods=1).mean(),
                   color=color, lw=2, label=label)
    axes[0,0].plot(df['episode'], df['avg_reward'],
                   color=color, alpha=0.15, lw=0.8)

    # Travel time
    axes[0,1].plot(df['episode'],
                   df['avg_travel_time'].rolling(w, min_periods=1).mean(),
                   color=color, lw=2, label=label)
    axes[0,1].plot(df['episode'], df['avg_travel_time'],
                   color=color, alpha=0.15, lw=0.8)

    # DQN loss
    axes[1,0].plot(df['episode'],
                   df['avg_loss'].rolling(w, min_periods=1).mean(),
                   color=color, lw=2, label=label)

    # Buffer size
    axes[1,1].plot(df['episode'], df['buffer_size'],
                   color=color, lw=2, label=label)

titles   = ['Avg Reward per Episode', 'Avg Travel Time per Episode',
            'DQN Loss per Episode',   'Replay Buffer Size per Episode']
ylabels  = ['Avg Reward', 'Travel Time (s)', 'DQN Loss', 'Buffer Size (transitions)']
for ax, title, ylabel in zip(axes.flat, titles, ylabels):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Episode')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
save_path = '/content/dqn_all_stages.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

# Summary table
print('\nSummary — last 20 episodes of each stage:')
print(f'{"Stage":<30} {"Avg Reward":<14} {"Avg TT (s)":<14} {"Avg Loss"}')
print('-' * 70)
for ckpt_name, label, _ in STAGES:
    log_path = f'/content/{ckpt_name}/training_log.csv'
    if os.path.exists(log_path):
        df   = pd.read_csv(log_path)
        tail = df.tail(20)
        print(f'{label:<30} {tail["avg_reward"].mean():<14.4f} '
              f'{tail["avg_travel_time"].mean():<14.2f} '
              f'{tail["avg_loss"].mean():.6f}')